# NB02 · Semana 4 — Extracción de la API hacia la capa bronze

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno (a completar)**

## Objetivo

Construir el primer tramo del pipeline: **extraer** datos de la API del BCRA y depositarlos en la capa **bronze** del lake como tabla Delta, aplicando dos patrones clave:

- **Carga incremental**: traer solo lo nuevo desde la última corrida (*watermark*).
- **Idempotencia**: reejecutar el pipeline no duplica datos ni rompe nada.

> Este notebook es la base de la **Entrega 1 (E1)** del proyecto.

## 1. Configuración del acceso al lake

Las credenciales de MinIO vienen por variables de entorno del contenedor.

In [10]:
import os
BUCKET = os.environ.get("MINIO_BUCKET", "lake")        # servidor: grupoNN · local: lake
BRONZE = f"s3://{BUCKET}/bronze/series"
STORAGE_OPTIONS = {
    "AWS_ACCESS_KEY_ID":     os.environ.get("AWS_ACCESS_KEY_ID", "minioadmin"),
    "AWS_SECRET_ACCESS_KEY": os.environ.get("AWS_SECRET_ACCESS_KEY", "minioadmin"),
    "AWS_ENDPOINT_URL":      os.environ.get("AWS_ENDPOINT_URL", "http://minio:9000"),
    "AWS_ALLOW_HTTP":        "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}
print('Destino bronze:', BRONZE)

Destino bronze: s3://grupo07/bronze/series


## 2. Leer el *watermark*

El watermark es la máxima fecha ya cargada en bronze. Si la tabla no existe todavía, devolvemos `None` (primera corrida).

In [11]:
pip install deltalake

Note: you may need to restart the kernel to use updated packages.


In [12]:
from deltalake import DeltaTable

def leer_watermark():
    # Devolver el máximo 'id' ya presente en bronze, o None si no existe.
    try:
        df = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS).to_pandas()
        if df.empty:
            return None
        return int(df['id'].max())
    except Exception:
        return None

print('watermark actual:', leer_watermark())

watermark actual: 3953074


## 3. Extraer de la API del BCRA (carga incremental)

La API monetaria del BCRA (v3.0) expone series por `idVariable`, con parámetros `desde` y `hasta`. Documentación: https://www.bcra.gob.ar/BCRAyVos/catalogo-de-APIs-banco-central.asp

⚠️ **Validar en tu entorno.** Esta celda hace una llamada HTTP real a la API; requiere conexión desde tu red. El patrón (paginación por fecha, manejo de errores, filtrado por watermark) es el estándar; ajustá los `idVariable` a las series de tu interés.

In [13]:
import os, requests, pandas as pd

COLUMNAS_ESQUEMA = [
    'id', 'date_added', 'url', 'url_status', 'last_online', 
    'threat', 'tags', 'urlhaus_link', 'reporter'
]

def extraer_historico(desde=None):
    print("Descargando histórico completo de URLhaus (descomprimiendo el ZIP en memoria, paciencia)...")
    
    # Agregamos compression='zip' y limpiamos errores de encoding
    df = pd.read_csv(
        "https://urlhaus.abuse.ch/downloads/csv/", 
        comment='#', 
        names=COLUMNAS_ESQUEMA,
        compression='zip',
        encoding_errors='ignore',
        low_memory=False
    )
    
    # Forzamos los tipos por si alguna fila sucia superó el filtro de comentarios
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    df = df.dropna(subset=['id'])
    df['id'] = df['id'].astype('int64')
    
    df['date_added'] = pd.to_datetime(df['date_added'], format='mixed', errors='coerce', utc=True)
    return df.dropna(subset=['id', 'date_added'])

def extraer_de_api(desde=None):
    print(f"Consultando API reciente (desde watermark: {desde})...")
    # La Auth-Key se lee del entorno (dags/.env o variable del JupyterHub): nunca va escrita en el código
    clave = os.environ.get("URLHAUS_AUTH_KEY")
    if not clave:
        raise RuntimeError("Falta la variable de entorno URLHAUS_AUTH_KEY")
    headers = {"Auth-Key": clave}
    
    try:
        # Ahora sí le pasamos los headers a la petición
        res = requests.get("https://urlhaus-api.abuse.ch/v1/urls/recent/", headers=headers, timeout=10)
        res.raise_for_status() 
    except requests.exceptions.RequestException as e:
        print(f"Error HTTP al contactar URLhaus: {e}")
        return pd.DataFrame()
        
    urls = res.json().get("urls", [])
    if not urls: return pd.DataFrame()
        
    df = pd.DataFrame(urls).rename(columns={'urlhaus_reference': 'urlhaus_link'})
    if 'last_online' not in df.columns: df['last_online'] = None
        
    df = df[COLUMNAS_ESQUEMA].copy()
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    df = df.dropna(subset=['id'])
    df['id'] = df['id'].astype('int64')
    
    df['date_added'] = pd.to_datetime(df['date_added'], format='mixed', errors='coerce', utc=True)
    df = df.dropna(subset=['id', 'date_added'])
    
    if desde is not None:
        df = df[df['id'] > desde]
    return df

## 4. Escribir en bronze (Delta, append idempotente)

Primera corrida: crea la tabla. Corridas siguientes: agrega solo lo nuevo.

In [14]:
from deltalake import write_deltalake

def ingerir(extractor, modo='append'):
    # Si es carga histórica (overwrite), ignoramos el watermark. 
    # Si es incremental (append), leemos hasta dónde llegamos.
    wm = leer_watermark() if modo == 'append' else None
    nuevos = extractor(desde=wm)
    
    if nuevos is None or nuevos.empty:
        print(f"Sin novedades desde el watermark {wm} (0 filas).")
        return 0
        
    # Deduplicación local para garantizar idempotencia
    nuevos = nuevos.drop_duplicates(subset=['id'])
    
    write_deltalake(
        BRONZE,
        nuevos,
        mode=modo,
        schema_mode="overwrite" if modo == 'overwrite' else "merge",
        storage_options=STORAGE_OPTIONS
    )
    return len(nuevos)

## 5. Verificar idempotencia

Reejecutar `ingerir` no debe duplicar datos: la segunda corrida arranca de un watermark ya actualizado y no encuentra novedades.

In [15]:
!pip install pyarrow

In [16]:
print("--- PASO 1: Carga Inicial Masiva (Backfill Histórico) ---")
# Usamos overwrite para limpiar pruebas viejas y cargar la historia base de URLhaus
filas_hist = ingerir(extraer_historico, modo='overwrite')
print(f"Resultado Paso 1: {filas_hist} filas históricas ingeridas en MinIO.\\n")

print("--- PASO 2: Carga Incremental (Simulación ingesta diaria) ---")
# Usamos append y la API. Leerá el watermark gigante de la carga histórica.
# Si el CSV estaba muy al día, esto sumará las URLs de las últimas horas (o 0).
filas_inc = ingerir(extraer_de_api, modo='append')
print(f"Resultado Paso 2: {filas_inc} filas nuevas ingeridas desde la API.\\n")

print("--- PASO 3: Verificación de Idempotencia ---")
# Volvemos a correr la API inmediatamente. 
# Como el watermark ya subió en el Paso 2, sí o sí tiene que traer 0 filas.
filas_idem = ingerir(extraer_de_api, modo='append')
print(f"Resultado Paso 3: {filas_idem} filas ingeridas. ¡Idempotencia validada con éxito!")

--- PASO 1: Carga Inicial Masiva (Backfill Histórico) ---
Descargando histórico completo de URLhaus (descomprimiendo el ZIP en memoria, paciencia)...
Resultado Paso 1: 73378 filas históricas ingeridas en MinIO.\n
--- PASO 2: Carga Incremental (Simulación ingesta diaria) ---
Consultando API reciente (desde watermark: 3953116)...
Sin novedades desde el watermark 3953116 (0 filas).
Resultado Paso 2: 0 filas nuevas ingeridas desde la API.\n
--- PASO 3: Verificación de Idempotencia ---
Consultando API reciente (desde watermark: 3953116)...
Sin novedades desde el watermark 3953116 (0 filas).
Resultado Paso 3: 0 filas ingeridas. ¡Idempotencia validada con éxito!


## Entrega 1 (E1)

Lo que se evalúa: ingesta idempotente hacia bronze con carga incremental y manejo de errores documentado. El enunciado y la rúbrica completos se publican en la semana 4.